In [108]:
from analyses.spike_rate import compute_mean_spike_rate_for_cells, compute_mean_spike_rate_for_windows
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import zscore

# Load neural data
windows_df = pd.read_pickle('/old/old_analysis_cache/Zombies_significant_windows_pANOVAorGLM_passed.pkl')
spike_df = compute_mean_spike_rate_for_windows(windows_df)

Computing windowed mean spike rate: 100%|██████████| 525/525 [00:08<00:00, 63.67it/s]


In [125]:
# Load behavioral matrix
parent_dir = "/home/connorlab/Documents/GitHub/Julie/social_data/zombies_social_data/"
# Load data
aff_df = pd.read_excel(parent_dir + "zombies_feature_df_affiliation.xlsx", index_col=0)
sub_df = pd.read_excel(parent_dir + "zombies_feature_df_submission.xlsx", index_col=0)
ago_df = pd.read_excel(parent_dir + "zombies_feature_df_agonism.xlsx", index_col=0)

In [126]:
# 함수: "Behavior Towards 94B" → "94B"
def clean_col(colname):
    return colname.split()[-1]  # 마지막 단어만 추출

# 컬럼 이름 정리
aff_df.columns = [clean_col(c) for c in aff_df.columns]
sub_df.columns = [clean_col(c) for c in sub_df.columns]
ago_df.columns = [clean_col(c) for c in ago_df.columns]

# === Feature names for labeling ===
feature_names = (
    [f"Aff_{c}" for c in aff_df.columns] +
    [f"Sub_{c}" for c in sub_df.columns] +
    [f"Ago_{c}" for c in ago_df.columns]
)

# 공통 monkey 추출
monkeys = aff_df.index.intersection(sub_df.index).intersection(ago_df.index)
stimulus_monkeys= monkeys.drop("81G")
stimulus_monkeys

Index(['7124', '69X', '72X', '94B', '110E', '67G', '143H', '87J', '151J'], dtype='object', name='Focal Name')

In [127]:
# Build behavior vector per monkey (row-wise vector from each matrix)

# === Build X (behavior matrix) ===
X = np.array([
    np.concatenate([aff_df.loc[m], sub_df.loc[m], ago_df.loc[m]])
    for m in stimulus_monkeys
])
X

array([[ 0, 19,  9, 38, 84, 27, 13, 14,  4,  9,  0,  0,  0,  1,  0,  1,
         0,  0,  2,  0,  0,  4,  2,  7,  4,  7,  3,  9,  3,  2],
       [15,  0,  9, 41,  4, 13, 19, 71, 12,  0,  4,  0,  0, 13,  0,  0,
         0,  2,  1,  0,  0,  0,  0,  2,  1,  1,  8,  4,  2,  0],
       [18, 10,  0, 21,  7, 17, 49, 18,  3,  1,  9,  2,  0, 10,  0,  0,
         4,  3,  0,  0,  0,  1,  0,  0,  0,  7,  0,  1,  0,  2],
       [38, 43, 24,  0, 18,  6, 31, 26, 29,  4,  7,  0,  2,  0,  0,  0,
         0,  1,  6,  0,  1,  8,  0,  0,  9,  5,  6, 10, 11,  2],
       [90,  3,  8,  8,  0, 22,  8,  9,  1, 18, 11,  8,  3, 15,  0,  2,
         4,  2,  4, 18,  0,  0,  0,  0,  0,  1,  0,  1,  3, 10],
       [23, 12, 17,  1, 23,  0, 23, 16,  3, 10, 90, 16, 14, 43,  0,  0,
        10,  7,  9,  6,  0,  0,  0,  0,  2,  0,  0,  0,  3,  1],
       [11, 70, 18, 17,  8, 17, 33,  0,  5,  3, 41,  6,  1, 28,  0,  0,
         2,  0,  5,  0,  0,  1,  1,  0,  1,  0,  7,  0,  0,  1],
       [ 3,  6,  4, 31,  2,  1,  2,  4,  

In [132]:
# Build Y (still z-scored across monkeys per neuron for comparability)
# filtered_df = spike_df[(spike_df['MonkeyGroup'] == 'Zombies')]
filtered_df = spike_df[(spike_df['MonkeyGroup'] == 'Zombies') & (spike_df['NeuronID'].str.contains('Unit')) ]
filtered_df['NeuronID'].nunique()

46

In [129]:
value_table = filtered_df.groupby(['NeuronID', 'MonkeyName'])['MeanSpikeRate'].mean().unstack()
value_table = value_table.loc[:, stimulus_monkeys]
# Transpose for PLS (rows = monkeys, cols = neural features)
from scipy.stats import zscore
Y = pd.DataFrame(zscore(value_table.T, axis=0), index=stimulus_monkeys)
# Y_df = pd.DataFrame(value_table.T, index=stimulus_monkeys)
print(Y)

NeuronID    AMG_2023-09-26_1_Channel.C_014_Unit 1  \
Focal Name                                          
7124                                     1.425822   
69X                                      0.768670   
72X                                     -1.132641   
94B                                     -2.010778   
110E                                    -0.375809   
67G                                      0.030297   
143H                                     0.561926   
87J                                     -0.036157   
151J                                     0.768670   

NeuronID    AMG_2023-09-26_1_Channel.C_018_Unit 1  \
Focal Name                                          
7124                                     0.743919   
69X                                      0.656071   
72X                                      1.534551   
94B                                     -0.724397   
110E                                    -0.661649   
67G                                     -1.62

In [130]:
# Run PLS
from sklearn.preprocessing import StandardScaler

X_scaled = X  # no z-score, raw values
Y_scaled = StandardScaler().fit_transform(Y)  # standardize neural space

pls = PLSRegression(n_components=4)
pls.fit(X_scaled, Y_scaled)

PLSRegression(n_components=4)

In [119]:
## 3D PLOT
from mpl_toolkits.mplot3d import Axes3D  # Required for 3D plotting

scores = pls.x_scores_  # shape: (9, 3)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(scores[:, 0], scores[:, 1], scores[:, 2], s=100)

for i, name in enumerate(stimulus_monkeys):
    ax.text(scores[i, 0] + 0.02, scores[i, 1], scores[i, 2], name, fontsize=9)

ax.set_xlabel('PLS Component 1')
ax.set_ylabel('PLS Component 2')
ax.set_zlabel('PLS Component 3')
ax.set_title('Stimulus Monkeys in 3D PLS Component Space')

plt.tight_layout()
plt.show()

In [123]:
import matplotlib.pyplot as plt

scores = pls.x_scores_  # shape: (n_monkeys, n_components)

plt.figure(figsize=(8, 6))
plt.scatter(scores[:, 0], scores[:, 1], s=100, edgecolor='k')

# Annotate each monkey
for i, name in enumerate(stimulus_monkeys):
    plt.text(scores[i, 0] + 0.02, scores[i, 1], name, fontsize=9)

plt.axhline(0, color='gray', linewidth=1, linestyle='--')
plt.axvline(0, color='gray', linewidth=1, linestyle='--')
plt.xlabel("PLS Component 1")
plt.ylabel("PLS Component 2")
plt.title("Stimulus Monkeys in PLS Score Space (2D)")
plt.grid(True)
plt.tight_layout()
plt.show()


In [124]:
import seaborn as sns

weights_matrix = pls.x_weights_  # shape: (num_features, num_components)
sns.heatmap(weights_matrix,
            xticklabels=[f'Comp {i+1}' for i in range(weights_matrix.shape[1])],
            yticklabels=feature_names,
            cmap='coolwarm', center=0)

plt.title('Feature Weights Across PLS Components')
plt.xlabel('PLS Components')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

In [135]:
# Label behavioral variables
behavior_labels = (
    list(aff_df.columns) +
    list(sub_df.columns) +
    list(ago_df.columns)
)

# Behavioral loadings
loadings_df = pd.DataFrame(pls.x_loadings_, index=behavior_labels, columns=[f'PLS{i+1}' for i in range(pls.n_components)])

# Sort by loading magnitude on first component
loadings_df['abs_PLS1'] = loadings_df['PLS1'].abs()
loadings_df['abs_PLS2'] = loadings_df['PLS2'].abs()
loadings_df['abs_PLS3'] = loadings_df['PLS3'].abs()
loadings_df_sorted = loadings_df.sort_values(by='abs_PLS1', ascending=False)
loadings_df_sorted

,PLS1,PLS2,PLS3,PLS4,abs_PLS1,abs_PLS2,abs_PLS3
69X,0.336827,-0.109114,0.027784,0.052626,0.336827,0.109114,0.027784
72X,0.306319,-0.079699,0.137375,0.023746,0.306319,0.079699,0.137375
81G,0.299130,-0.152727,0.027407,0.175661,0.299130,0.152727,0.027407
7124,0.270815,-0.202742,0.136950,-0.007037,0.270815,0.202742,0.136950
94B,0.258351,-0.245826,0.070220,0.034307,0.258351,0.245826,0.070220
151J,0.256422,0.194456,-0.180741,0.077154,0.256422,0.194456,0.180741
143H,0.254330,-0.158422,0.096637,0.302973,0.254330,0.158422,0.096637
87J,0.253790,0.041714,0.241907,-0.185900,0.253790,0.041714,0.241907
94B,-0.239268,-0.122308,-0.163998,-0.006462,0.239268,0.122308,0.163998
67G,-0.229449,0.139439,-0.050041,0.316149,0.229449,0.139439,0.050041


In [106]:
# === Top behavioral features per component ===
loadings_df = pd.DataFrame(pls.x_loadings_, index=feature_names, columns=[f'PLS{i+1}' for i in range(pls.n_components)])
for i in range(3):
    print(f"\nTop 10 features for PLS{i+1}:")
    display(loadings_df.iloc[:, i].abs().sort_values(ascending=False).head(10))


Top 10 features for PLS1:


Sub_69X     0.336827
Sub_72X     0.306319
Sub_81G     0.299130
Sub_7124    0.270815
Sub_94B     0.258351
Sub_151J    0.256422
Sub_143H    0.254330
Sub_87J     0.253790
Aff_94B     0.239268
Ago_67G     0.229449
Name: PLS1, dtype: float64


Top 10 features for PLS2:


Ago_87J     0.296745
Ago_69X     0.292293
Ago_7124    0.282826
Ago_143H    0.282338
Sub_67G     0.273859
Aff_7124    0.272441
Ago_151J    0.270507
Ago_110E    0.246502
Sub_94B     0.245826
Aff_151J    0.244620
Name: PLS2, dtype: float64


Top 10 features for PLS3:


Aff_87J     0.320564
Aff_72X     0.317857
Ago_7124    0.294186
Sub_67G     0.291027
Ago_110E    0.259398
Ago_69X     0.258880
Ago_151J    0.250019
Ago_87J     0.248418
Sub_87J     0.241907
Aff_69X     0.223163
Name: PLS3, dtype: float64

In [133]:
# === Create labels for neural features ===
# Each row of value_table is a neuron — use those as labels
neuron_labels = value_table.index.tolist()  # (NeuronID)

# === Neural loadings DataFrame ===
neural_loadings_df = pd.DataFrame(
    pls.y_loadings_,
    index=neuron_labels,
    columns=[f'PLS{i+1}' for i in range(pls.n_components)]
)

# === Top contributing neurons per component ===
for i in range(3):  # or range(pls.n_components)
    print(f"\n🔍 Top 10 neurons for PLS component {i+1}:")
    display(
        neural_loadings_df.iloc[:, i]
        .abs()
        .sort_values(ascending=False)
        .head(10)
    )




🔍 Top 10 neurons for PLS component 1:


Unknown_2023-12-18_2_Channel.C_002_Unit 1    0.279990
Unknown_2023-12-18_1_Channel.C_008_Unit 1    0.267946
AMG_2023-09-26_1_Channel.C_018_Unit 1        0.267204
AMG_2023-10-04_1_Channel.C_019_Unit 1        0.231371
AMG_2023-10-04_3_Channel.C_009_Unit 3        0.230087
AMG_2023-09-26_1_Channel.C_018_Unit 2        0.187001
AMG_2023-10-04_2_Channel.C_009_Unit 1        0.180005
Unknown_2023-12-18_2_Channel.C_017_Unit 1    0.175830
Unknown_2023-12-18_1_Channel.C_014_Unit 1    0.174342
AMG_2023-09-26_1_Channel.C_027_Unit 1        0.165808
Name: PLS1, dtype: float64


🔍 Top 10 neurons for PLS component 2:


AMG_2023-10-04_4_Channel.C_021_Unit 1    0.378492
AMG_2023-09-26_1_Channel.C_020_Unit 1    0.319456
AMG_2023-10-04_4_Channel.C_027_Unit 1    0.314222
AMG_2023-10-04_4_Channel.C_025_Unit 1    0.296244
AMG_2023-10-04_2_Channel.C_029_Unit 1    0.287094
AMG_2023-09-26_3_Channel.C_027_Unit 2    0.274541
AMG_2023-09-26_1_Channel.C_018_Unit 2    0.271529
AMG_2023-09-26_1_Channel.C_027_Unit 1    0.251145
AMG_2023-10-05_2_Channel.C_009_Unit 2    0.249199
AMG_2023-10-04_2_Channel.C_022_Unit 1    0.243938
Name: PLS2, dtype: float64


🔍 Top 10 neurons for PLS component 3:


AMG_2023-10-04_4_Channel.C_018_Unit 2        0.314589
AMG_2023-10-04_4_Channel.C_027_Unit 1        0.258407
AMG_2023-10-04_4_Channel.C_022_Unit 2        0.252305
AMG_2023-10-04_3_Channel.C_009_Unit 3        0.242678
AMG_2023-09-28_1_Channel.C_025_Unit 1        0.223345
AMG_2023-09-28_1_Channel.C_025_Unit 2        0.209035
Unknown_2023-12-18_3_Channel.C_021_Unit 1    0.208492
AMG_2023-09-26_1_Channel.C_018_Unit 1        0.200419
AMG_2023-10-04_2_Channel.C_022_Unit 1        0.199369
ER_2023-11-28_4_Channel.C_024_Unit 1         0.190669
Name: PLS3, dtype: float64

In [134]:

plt.figure(figsize=(10, len(neuron_labels)*0.2))
sns.heatmap(
    neural_loadings_df,
    xticklabels=True,
    yticklabels=neuron_labels,
    cmap='coolwarm', center=0
)
plt.title("Neural Loadings on PLS Components")
plt.tight_layout()
plt.show()